In [18]:
import cv2 as cv
import numpy as np
import scipy.ndimage as ndi
import os

# --- Step 0: Load the Image ---
root = os.getcwd()
img_path = os.path.join(root, "hpref_cropped.tif")
img_gray = cv.imread(img_path, cv.IMREAD_GRAYSCALE)

if img_gray is None:
    print("Error: Could not load image. Make sure 'hpref.tif' is in the same directory.")
else:
    print(f"Successfully loaded image 'hpref.tif' with shape {img_gray.shape}")

    # --- Step 1: Isolate the Spots (Get Binary Mask) ---
    ret, binary_mask = cv.threshold(img_gray, 50, 255, cv.THRESH_BINARY)
    print("Created binary mask from image.")

    # --- Step 2: Find and Label All Spots (with Stats) ---
    # --- CHANGED ---
    # We use 'cv.connectedComponentsWithStats' to get the size of each spot
    # 'stats' is a matrix where stats[label, cv.CC_STAT_AREA] is the pixel count.
    num_labels, labels_matrix, stats, centroids = cv.connectedComponentsWithStats(binary_mask)
    
    print(f"Found {num_labels - 1} total blobs (including noise).")

    # --- NEW: Step 2.5: Filter out Noise ---
    # We will build a new list of labels, keeping only the ones that are "real" spots.
    
    # Set a minimum pixel size to be considered a "real" spot.
    # You can adjust this value if needed.
    min_spot_size = 50  
    
    valid_indices = []
    for label_id in range(1, num_labels): # Loop from 1 (skip background 0)
        # Check the size (area) of the blob with this label
        if stats[label_id, cv.CC_STAT_AREA] > min_spot_size:
            valid_indices.append(label_id)
            
    print(f"Kept {len(valid_indices)} spots after filtering blobs smaller than {min_spot_size} pixels.")

    # --- Step 3: Calculate Precise Intensity-Weighted Centroids ---
    # --- CHANGED ---
    # We now pass our *clean list* of 'valid_indices' to the 'index' parameter.
    # It will only calculate centroids for the labels in this list.
    precise_centroids = ndi.center_of_mass(img_gray, labels=labels_matrix, index=valid_indices)
    
    # Note: ndi.center_of_mass returns (y, x) coordinates.
    print(f"Calculated {len(precise_centroids)} precise sub-pixel centroids.")


    # --- Step 4: Organize and Save the Centroids ---
    
    # Convert [(y1, x1), (y2, x2)...] to [(x1, y1), (x2, y2)...]
    centroids_xy = [(coord[1], coord[0]) for coord in precise_centroids]
    
    # Sort the list:
    # key=lambda c: (c[1], c[0]) means "sort by Y (c[1]) first, then by X (c[0])"
    sorted_centroids = sorted(centroids_xy, key=lambda c: (c[1], c[0]))
    
    print("Sorted all centroids (top-to-bottom, left-to-right).")
    
    # --- Step 5: Save to File ---
    file_name = "centros_Yobod.txt" # or "output_organized.txt"
    
    with open(file_name, 'w') as f:
        for i, (x_coord, y_coord) in enumerate(sorted_centroids):
            label = i + 1
            f.write(f"{label} {x_coord:.8f} {y_coord:.8f}\n")

    print(f"Successfully saved {len(sorted_centroids)} precise, sorted centroids to {file_name}")

Successfully loaded image 'hpref.tif' with shape (480, 481)
Created binary mask from image.
Found 274 total blobs (including noise).
Kept 1 spots after filtering blobs smaller than 50 pixels.
Calculated 1 precise sub-pixel centroids.
Sorted all centroids (top-to-bottom, left-to-right).
Successfully saved 1 precise, sorted centroids to centros_Yobod.txt


In [19]:
import cv2 as cv
import numpy as np
import os

# -------------------------------------------------------------------
# ### TUNE THIS VALUE ###
# Start with 50. If you get too many blobs, try 60. If too few, try 40.
# Your goal is to find a value that gives you at least 253 blobs.
threshold_value = 50
# -------------------------------------------------------------------

# --- Load Image ---
root = os.getcwd()
# Using the filename from your code
img_path = os.path.join(root, "hpref_cropped.tif") 
img_gray = cv.imread(img_path, cv.IMREAD_GRAYSCALE)

if img_gray is None:
    print(f"Error: Could not load image 'hpref_cropped.tif'.")
else:
    print(f"Successfully loaded image 'hpref_cropped.tif' with shape {img_gray.shape}")
    
    # --- Get Binary Mask ---
    ret, binary_mask = cv.threshold(img_gray, threshold_value, 255, cv.THRESH_BINARY)
    
    # --- Get Stats for ALL Blobs ---
    num_labels, labels_matrix, stats, centroids = cv.connectedComponentsWithStats(binary_mask)
    
    # Get the size (area) of all blobs, skipping the background (label 0)
    all_blob_sizes = stats[1:, cv.CC_STAT_AREA]
    
    # Sort the sizes from largest to smallest
    all_blob_sizes_sorted = sorted(all_blob_sizes, reverse=True)
    
    # --- Print the sizes ---
    print(f"Found {len(all_blob_sizes_sorted)} total blobs with threshold={threshold_value}.")
    print("Here are their sizes, from largest to smallest:")
    
    # Print all sizes, with a marker at #253
    for i, size in enumerate(all_blob_sizes_sorted):
        print(f"Blob #{i+1}: {size} pixels")
        if i == 252: # This is the 253rd blob
            print("---------------------------------")
            print("--- THIS IS YOUR 253rd SPOT ---")
            print(f"--- ITS SIZE IS {size} PIXELS ---")
            print("---------------------------------")
            
    # Give advice
    if len(all_blob_sizes_sorted) < 253:
        print("\n*** ACTION NEEDED: You found less than 253 spots. ***")
        print("*** Try lowering the 'threshold_value' (e.g., to 40) and run again. ***")
    else:
        size_of_253rd_spot = all_blob_sizes_sorted[252] # Index 252 is the 253rd item
        size_of_254th_blob = all_blob_sizes_sorted[253] # Index 253 is the 254th item
        
        # Calculate a suggested new min_spot_size
        suggested_min_size = (size_of_253rd_spot + size_of_254th_blob) / 2
        
        print(f"\n*** SUCCESS! You found all 253 spots (and {len(all_blob_sizes_sorted) - 253} noise blobs). ***")
        print(f"The 253rd spot has size {size_of_253rd_spot}.")
        print(f"The 254th (noise) blob has size {size_of_254th_blob}.")
        print(f"A perfect 'min_spot_size' for your final code would be between these two values.")
        print(f"*** Use this value for min_spot_size in Step 2: {int(suggested_min_size)} ***")

Successfully loaded image 'hpref_cropped.tif' with shape (480, 481)
Found 274 total blobs with threshold=50.
Here are their sizes, from largest to smallest:
Blob #1: 52 pixels
Blob #2: 46 pixels
Blob #3: 45 pixels
Blob #4: 45 pixels
Blob #5: 45 pixels
Blob #6: 44 pixels
Blob #7: 42 pixels
Blob #8: 42 pixels
Blob #9: 41 pixels
Blob #10: 41 pixels
Blob #11: 40 pixels
Blob #12: 40 pixels
Blob #13: 39 pixels
Blob #14: 39 pixels
Blob #15: 39 pixels
Blob #16: 38 pixels
Blob #17: 38 pixels
Blob #18: 38 pixels
Blob #19: 38 pixels
Blob #20: 38 pixels
Blob #21: 38 pixels
Blob #22: 38 pixels
Blob #23: 37 pixels
Blob #24: 37 pixels
Blob #25: 37 pixels
Blob #26: 37 pixels
Blob #27: 37 pixels
Blob #28: 36 pixels
Blob #29: 36 pixels
Blob #30: 36 pixels
Blob #31: 36 pixels
Blob #32: 35 pixels
Blob #33: 35 pixels
Blob #34: 35 pixels
Blob #35: 35 pixels
Blob #36: 35 pixels
Blob #37: 35 pixels
Blob #38: 35 pixels
Blob #39: 34 pixels
Blob #40: 34 pixels
Blob #41: 34 pixels
Blob #42: 34 pixels
Blob #43: 34

In [23]:
import cv2 as cv
import numpy as np
import scipy.ndimage as ndi
import os

# -------------------------------------------------------------------
# ### SET THESE VALUES BASED ON YOUR STEP 1 DIAGNOSTIC ###
#
# (The value you found that gives you at least 253 blobs)
threshold_value = 60
#
# (The value suggested by the diagnostic code, e.g., 150)
min_spot_size = 5  
# -------------------------------------------------------------------


# --- Step 0: Load the Image ---
root = os.getcwd()
img_path = os.path.join(root, "hpref_cropped.tif")
img_gray = cv.imread(img_path, cv.IMREAD_GRAYSCALE)

if img_gray is None:
    print("Error: Could not load image.")
else:
    print(f"Successfully loaded image 'hpref_cropped.tif' with shape {img_gray.shape}")

    # --- Step 1: Isolate the Spots (Get Binary Mask) ---
    ret, binary_mask = cv.threshold(img_gray, threshold_value, 255, cv.THRESH_BINARY)
    print(f"Created binary mask with threshold={threshold_value}.")

    # --- Step 2: Find and Label All Spots (with Stats) ---
    num_labels, labels_matrix, stats, centroids = cv.connectedComponentsWithStats(binary_mask)
    print(f"Found {num_labels - 1} total blobs (including noise).")

    # --- Step 2.5: Filter out Noise ---
    valid_indices = []
    for label_id in range(1, num_labels):
        if stats[label_id, cv.CC_STAT_AREA] > min_spot_size:
            valid_indices.append(label_id)
            
    print(f"Kept {len(valid_indices)} spots after filtering blobs smaller than {min_spot_size} pixels.")

    # --- Step 3: Calculate Precise Intensity-Weighted Centroids ---
    precise_centroids = ndi.center_of_mass(img_gray, labels=labels_matrix, index=valid_indices)
    print(f"Calculated {len(precise_centroids)} precise sub-pixel centroids.")

    # --- Step 4: Organize and Save the Centroids ---
    centroids_xy = [(coord[1], coord[0]) for coord in precise_centroids]
    sorted_centroids = sorted(centroids_xy, key=lambda c: (c[1], c[0]))
    print("Sorted all centroids (top-to-bottom, left-to-right).")
    
    # --- Step 5: Save to File ---
    file_name = "centros_Yobod.txt" 
    
    with open(file_name, 'w') as f:
        for i, (x_coord, y_coord) in enumerate(sorted_centroids):
            label = i + 1
            f.write(f"{label} {x_coord:.8f} {y_coord:.8f}\n")

    print(f"Successfully saved {len(sorted_centroids)} precise, sorted centroids to {file_name}")

Successfully loaded image 'hpref_cropped.tif' with shape (480, 481)
Created binary mask with threshold=60.
Found 268 total blobs (including noise).
Kept 252 spots after filtering blobs smaller than 5 pixels.
Calculated 252 precise sub-pixel centroids.
Sorted all centroids (top-to-bottom, left-to-right).
Successfully saved 252 precise, sorted centroids to centros_Yobod.txt
